# 제16장. AI 에이전트와 기획 워크플로우
## 3시간 인터랙티브 강의 (이론 + 실습)

**학습 목표:**
1. **프롬프트 엔지니어링**의 핵심 원칙(역할/맥락/지시/형식)을 이해하고 구조화된 프롬프트를 작성할 수 있다
2. **Chain-of-Thought**, **Few-Shot** 등 고급 프롬프팅 기법을 적용할 수 있다
3. **AI 에이전트**의 개념과 **ReAct 패턴**을 이해한다
4. **RAG(Retrieval-Augmented Generation)** 아키텍처를 이해하고 활용할 수 있다
5. **Human-in-the-Loop** 기반 인간-AI 협업 모델을 설계할 수 있다

**강의 구조:**

| 시간 | Part | 내용 |
|------|------|------|
| | **Part 1** | 📖 프롬프트 엔지니어링: 구조화된 프롬프트, 역할/맥락/지시/형식 + 조사 과제 |
| | **Part 2** | 📖 고급 프롬프팅과 AI 에이전트: CoT, Few-Shot, ReAct, Plan-Execute + 조사 과제 |
| | **휴식** | ☕ 15분 휴식 |
| | **Part 3** | 📖 RAG와 인간-AI 협업: 벡터 DB, 검색 전략, HITL, 윤리적 고려 + 조사 과제 |
| | **Part 4** | 🔬 실습: 프롬프트 품질 평가와 CoT/Few-Shot 구현 |
| | **Part 5** | 🔬 실습: 기획 워크플로우 시뮬레이션 |

---

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---
## Part 1: 프롬프트 엔지니어링 기초

### 16.1 효과적인 프롬프트 구조

AI를 기획 업무에 활용하는 첫 단계는 **효과적인 프롬프트 작성**이다. 프롬프트 엔지니어링(Prompt Engineering)은 대규모 언어 모델(LLM)에게 원하는 출력을 얻기 위해 입력을 설계하는 기술이다. 동일한 모델이라도 프롬프트에 따라 출력 품질이 **크게** 달라진다.

### 기본 프롬프트 vs 구조화된 프롬프트

| 유형 | 예시 | 문제점/장점 |
|------|------|-------------|
| **기본** | "시장 분석해줘" | 모호, 맥락 부재, 형식 미지정 |
| **구조화** | 역할+맥락+지시+제약+형식 포함 | 구체적, 일관된 품질, 재현 가능 |

### 표 16.1 프롬프트 5대 구성 요소

| 구성 요소 | 목적 | 예시 |
|-----------|------|------|
| **역할(Role)** | 전문성 맥락 부여 | "당신은 10년 경력의 전략 컨설턴트입니다" |
| **맥락(Context)** | 배경 정보 제공 | "시장 규모 2조원, 주요 플레이어 3사..." |
| **지시사항(Instruction)** | 수행할 작업 명시 | "다음 3가지 관점에서 분석하세요" |
| **제약조건(Constraints)** | 출력 범위 설정 | "데이터 기반 분석, 불확실 시 가정 명시" |
| **출력 형식(Format)** | 결과 형태 지정 | "JSON 형식, 표로 정리" |

### 역할 설정의 중요성

역할 설정은 단순한 형식이 아니라, AI의 응답 스타일, 전문 용어 사용, 분석 깊이에 **직접적인 영향**을 미친다:
- **"당신은 신입사원입니다"** → 기초적 설명, 일반 용어
- **"당신은 10년 경력 컨설턴트입니다"** → 심층 분석, 전문 용어, 구조화된 프레임워크
- **"당신은 업계 전문 기자입니다"** → 트렌드 중심, 이해하기 쉬운 설명

### 맥락 제공의 4대 원칙

1. **구체적 수치**: "시장이 성장 중" → "시장 규모 2조원, CAGR 25%"
2. **시간적 맥락**: "2024년 기준", "향후 3년간"
3. **조직 상황**: "중소기업으로서", "시장 점유율 3위"
4. **이해관계자 명시**: "CEO 보고용", "투자위원회 검토용"

In [ ]:
# === 프롬프트 품질 평가 시스템 ===

def evaluate_prompt_quality(prompt_text):
    """프롬프트 품질을 5개 차원으로 평가 (시뮬레이션)"""
    scores = {}
    scores['Role'] = 5 if any(kw in prompt_text for kw in ['역할', '당신은', 'You are', '전문가', '컨설턴트']) else 1
    scores['Context'] = 5 if any(kw in prompt_text for kw in ['시장 규모', '배경', '현재', 'context', '조원', 'CAGR']) else 1
    scores['Instruction'] = 5 if any(kw in prompt_text for kw in ['분석하세요', '관점에서', 'analyze', '평가하세요', '도출하세요']) else 2
    scores['Constraints'] = 5 if any(kw in prompt_text for kw in ['제약', '기반', '제한', 'constraint', '데이터 기반', '가정 명시']) else 1
    scores['Format'] = 5 if any(kw in prompt_text for kw in ['표', 'JSON', '형식', 'format', '목록으로']) else 1
    return scores

# 기본 프롬프트 vs 구조화된 프롬프트
basic_prompt = "시장 분석해줘"

structured_prompt = """
[역할] 당신은 10년 경력의 전략 컨설턴트입니다.
[맥락] 국내 전기차 충전 인프라 시장 규모 2조원, CAGR 25%.
       분석 주체: 중견 에너지 기업 (매출 5천억, 충전 사업 미경험)
[지시사항] 다음 3가지 관점에서 분석하세요: 1) 성장 동인, 2) 경쟁 구도, 3) 기회 영역
[제약조건] 데이터 기반 분석 우선. 불확실한 정보는 가정 명시.
[출력 형식] 각 관점별 핵심 발견을 표 형식으로 정리
"""

basic_scores = evaluate_prompt_quality(basic_prompt)
structured_scores = evaluate_prompt_quality(structured_prompt)

categories = list(basic_scores.keys())

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=list(basic_scores.values()) + [list(basic_scores.values())[0]],
    theta=categories + [categories[0]],
    fill='toself', name='Basic Prompt',
    line_color='#EF553B', fillcolor='rgba(239, 85, 59, 0.2)'
))
fig.add_trace(go.Scatterpolar(
    r=list(structured_scores.values()) + [list(structured_scores.values())[0]],
    theta=categories + [categories[0]],
    fill='toself', name='Structured Prompt',
    line_color='#636EFA', fillcolor='rgba(99, 110, 250, 0.2)'
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 6])),
    title='Prompt Quality Comparison: Basic vs Structured',
    showlegend=True, width=700, height=500
)
fig.show()

basic_total = sum(basic_scores.values())
structured_total = sum(structured_scores.values())
improvement = (structured_total - basic_total) / basic_total * 100

print(f"\n📊 프롬프트 품질 평가 결과")
print(f"{'='*50}")
print(f"기본 프롬프트 총점:     {basic_total}/25점")
print(f"구조화된 프롬프트 총점: {structured_total}/25점")
print(f"{'='*50}")
print(f"💡 구조화된 프롬프트가 기본 대비 {improvement:.0f}% 높은 품질 점수를 기록했습니다.")
print(f"\n   5가지 요소(역할/맥락/지시/제약/형식)를 모두 포함하면")
print(f"   AI 출력의 정확성, 구체성, 일관성이 크게 향상됩니다.")

In [ ]:
# === 역할별 응답 품질 차이 시뮬레이션 ===

roles = {
    'Junior\n(신입사원)': {'Depth': 2.1, 'Expertise': 1.8, 'Actionability': 1.5, 'Structure': 2.0, 'Insight': 1.7},
    'Expert\n(10년 컨설턴트)': {'Depth': 4.8, 'Expertise': 4.9, 'Actionability': 4.5, 'Structure': 4.7, 'Insight': 4.6},
    'Journalist\n(전문 기자)': {'Depth': 3.5, 'Expertise': 3.2, 'Actionability': 2.8, 'Structure': 4.2, 'Insight': 3.8}
}

dimensions = ['Depth', 'Expertise', 'Actionability', 'Structure', 'Insight']
colors = ['#EF553B', '#636EFA', '#00CC96']

fig = go.Figure()
for i, (role, scores) in enumerate(roles.items()):
    fig.add_trace(go.Bar(
        name=role, x=dimensions,
        y=[scores[d] for d in dimensions],
        marker_color=colors[i],
        text=[f"{scores[d]:.1f}" for d in dimensions],
        textposition='auto'
    ))

fig.update_layout(
    title='Response Quality by Role Setting',
    xaxis_title='Quality Dimension', yaxis_title='Score (1-5)',
    yaxis=dict(range=[0, 5.5]), barmode='group',
    width=800, height=500
)
fig.show()

print("💡 해석: 역할 설정에 따른 응답 품질 차이")
print("="*50)
print("• 전문가(Expert) 역할: 모든 차원에서 최고 점수 (평균 4.7)")
print("• 기자(Journalist) 역할: 구조성이 높지만 실행 가능성은 중간")
print("• 신입(Junior) 역할: 전반적으로 낮은 점수 (평균 1.8)")
print("\n→ 기획 업무에서는 '10년 경력 전략 컨설턴트' 등 전문가 역할을 설정하면")
print("  분석 깊이, 전문성, 실행 가능성이 모두 향상됩니다.")

### 📝 이론 과제 16-1 (15분)

#### 과제: 프롬프트 엔지니어링 기초

**질문:**

1. **구조화된 프롬프트의 5가지 요소**(역할/맥락/지시사항/제약조건/출력형식)를 각각 2-3문장으로 설명하고, 각 요소가 없을 때 발생하는 문제점을 서술하세요.

2. **역할 설정이 AI 출력에 미치는 영향**을 구체적으로 설명하세요. "신입사원"과 "10년 경력 컨설턴트" 역할의 차이가 왜 발생하는지 LLM의 작동 원리 관점에서 3-4문장으로 서술하세요.

3. **본인 업무(또는 관심 분야)에 사용할 프롬프트 1개**를 5가지 요소를 모두 포함하여 작성하세요. 각 요소를 `[역할]`, `[맥락]`, `[지시사항]`, `[제약조건]`, `[출력형식]` 태그로 구분하세요.

**조사 키워드:**
- "prompt engineering best practices 2024"
- "role prompting LLM"
- "structured prompt template"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 16-1 제출란

**학번:** ___________  
**이름:** ___________

#### 1. 구조화된 프롬프트 5가지 요소 설명

(여기에 작성)

#### 2. 역할 설정이 AI 출력에 미치는 영향

(여기에 작성)

#### 3. 본인 업무용 프롬프트 (5요소 포함)

```
[역할] 
[맥락] 
[지시사항] 
[제약조건] 
[출력형식] 
```

---
## Part 2: 고급 프롬프팅과 AI 에이전트

### 16.2 고급 프롬프팅 기법

기본 프롬프트 구조를 넘어, 복잡한 문제 해결을 위한 **고급 기법**들이 있다.

### 표 16.2 고급 프롬프팅 기법 비교

| 기법 | 핵심 아이디어 | 적합 용도 | 복잡도 |
|------|---------------|-----------|--------|
| **Chain-of-Thought (CoT)** | 단계별 추론 유도 | 수학, 논리, 재무 분석 | 중간 |
| **Few-Shot Learning** | 예시 기반 패턴 학습 | 분류, 형식화, 리스크 평가 | 낮음 |
| **Self-Consistency** | 다수결 앙상블 | 신뢰도 향상, 검증 | 높음 |
| **Tree of Thoughts** | 분기 탐색 | 복잡한 전략적 의사결정 | 높음 |
| **Prompt Chaining** | 순차적 분해 | 다단계 기획 작업 | 중간 |
| **Role-Play Debate** | 다관점 토론 | 의사결정 검증, 리스크 도출 | 중간 |

### 16.3 AI 에이전트란?

AI 에이전트는 **목표 달성을 위해 환경과 상호작용하는 자율적 시스템**이다. 단순 챗봇이 즉각 답변하는 반면, 에이전트는:

1. **목표 설정**: 사용자 요청을 이해하고 달성 목표를 정의
2. **계획 수립**: 목표 달성을 위한 단계별 계획 작성
3. **도구 활용**: 검색, 계산, DB 조회 등 외부 도구 호출
4. **결과 관찰**: 도구 실행 결과를 분석
5. **적응**: 결과에 따라 계획 수정 또는 다음 단계 진행

### 에이전트 패턴 비교

| 패턴 | 접근법 | 계획 방식 | 유연성 | 적합 용도 |
|------|--------|-----------|--------|----------|
| **ReAct** | Thought-Action-Observation 루프 | 암시적 | 높음 | 탐색적 분석 |
| **Plan-Execute** | 계획 수립 후 순차 실행 | 명시적 | 중간 | 구조화된 프로젝트 |
| **Reflexion** | 실행-반성-개선 반복 | 반복적 | 높음 | 품질 개선 |
| **Multi-Agent** | 역할별 에이전트 협업 | 분산 | 매우 높음 | 복잡한 기획 프로젝트 |

In [ ]:
# === Chain-of-Thought (CoT) 시뮬레이션 ===

print("=" * 60)
print("📋 질문: A기업의 수익성이 개선되었는가?")
print("   이전: 매출 100억, 비용 80억")
print("   현재: 매출 150억, 비용 100억")
print("=" * 60)

print("\n❌ Direct Answer (CoT 없음):")
print("   → '네, 매출이 100억에서 150억으로 증가하여 수익성이 개선되었습니다.'")
print("   (매출 증가 ≠ 수익성 개선, 불완전한 분석)")

steps = {
    'Step 1: Previous Profit': {'calc': '100 - 80 = 20', 'result': 20},
    'Step 2: Current Profit': {'calc': '150 - 100 = 50', 'result': 50},
    'Step 3: Previous Margin': {'calc': '20/100 = 20%', 'result': 20.0},
    'Step 4: Current Margin': {'calc': '50/150 = 33.3%', 'result': 33.3},
}

print("\n✅ Chain-of-Thought (단계별 추론):")
for step_name, info in steps.items():
    print(f"   {step_name}")
    print(f"   계산: {info['calc']}")
    unit = '억원' if info['result'] > 10 else '%'
    print(f"   결과: {info['result']}{unit}\n")

print("   Step 5: 결론")
print("   이익률이 20% → 33.3%로 개선되었으므로, 수익성이 개선되었다.")
print("   (단순 매출 증가가 아닌 이익률 기준으로 판단)")

fig = make_subplots(rows=1, cols=2, subplot_titles=('Revenue & Cost Comparison', 'Profit Margin Improvement'))

categories_x = ['Previous', 'Current']
fig.add_trace(go.Bar(name='Revenue', x=categories_x, y=[100, 150], marker_color='#636EFA',
                     text=['100B', '150B'], textposition='auto'), row=1, col=1)
fig.add_trace(go.Bar(name='Cost', x=categories_x, y=[80, 100], marker_color='#EF553B',
                     text=['80B', '100B'], textposition='auto'), row=1, col=1)
fig.add_trace(go.Bar(name='Profit', x=categories_x, y=[20, 50], marker_color='#00CC96',
                     text=['20B', '50B'], textposition='auto'), row=1, col=1)

fig.add_trace(go.Bar(
    name='Margin', x=['Previous Margin', 'Improvement', 'Current Margin'],
    y=[20, 13.3, 33.3], marker_color=['#636EFA', '#00CC96', '#AB63FA'],
    text=['20.0%', '+13.3%p', '33.3%'], textposition='auto', showlegend=False
), row=1, col=2)

fig.update_layout(title='CoT Analysis: Profitability Assessment', width=900, height=450, barmode='group')
fig.update_yaxes(title_text='Amount (Billion KRW)', row=1, col=1)
fig.update_yaxes(title_text='Profit Margin (%)', row=1, col=2)
fig.show()

print("\n💡 CoT의 핵심 가치: 추론 과정이 투명하여 검증 가능하다.")
print("   기획에서 재무 분석, 시나리오 평가 등 논리적 추론이 필요한 작업에 효과적이다.")

In [ ]:
# === Few-Shot 리스크 분류 시뮬레이션 ===

print("📋 Few-Shot Learning: 예시 기반 리스크 분류")
print("="*60)

examples = [
    {"input": "핵심 개발자 3명이 경쟁사로 이직했습니다.", "category": "인적자원 리스크", "severity": "높음"},
    {"input": "주요 원자재 가격이 30% 상승했습니다.", "category": "재무/운영 리스크", "severity": "중간"},
    {"input": "신규 개인정보보호법이 시행됩니다.", "category": "규제/컴플라이언스 리스크", "severity": "중간"},
]

print("\n[학습 예시 3건]")
for i, ex in enumerate(examples, 1):
    print(f"  예시 {i}: {ex['input']}")
    print(f"         → 카테고리: {ex['category']} / 심각도: {ex['severity']}")

test_cases = [
    {"input": "주요 고객사가 계약 해지를 통보했습니다.", "expected_cat": "고객/매출 리스크", "expected_sev": "높음"},
    {"input": "경쟁사가 가격을 40% 인하했습니다.", "expected_cat": "경쟁 리스크", "expected_sev": "높음"},
    {"input": "서버 장애로 서비스가 4시간 중단되었습니다.", "expected_cat": "기술/운영 리스크", "expected_sev": "중간"},
]

predictions = [
    {"category": "고객/매출 리스크", "severity": "높음", "confidence": 0.92},
    {"category": "경쟁 리스크", "severity": "높음", "confidence": 0.88},
    {"category": "기술/운영 리스크", "severity": "중간", "confidence": 0.85},
]

print("\n[분류 결과 3건]")
correct = 0
results_data = []
for i, (test, pred) in enumerate(zip(test_cases, predictions), 1):
    match = pred['category'] == test['expected_cat']
    correct += int(match)
    status = "✅" if match else "❌"
    print(f"  입력 {i}: {test['input']}")
    print(f"  예측: {pred['category']} / {pred['severity']} (신뢰도: {pred['confidence']:.0%}) {status}")
    results_data.append({'input_id': f'Test {i}', 'category': pred['category'],
                         'severity': pred['severity'], 'confidence': pred['confidence']})

accuracy = correct / len(test_cases)
print(f"\n📊 분류 정확도: {accuracy:.0%} ({correct}/{len(test_cases)})")

categories_list = [r['category'] for r in results_data]
confidences = [r['confidence'] for r in results_data]
severities = [r['severity'] for r in results_data]
colors_bar = ['#EF553B' if s == '높음' else '#FFA15A' for s in severities]

fig = go.Figure(go.Bar(
    x=categories_list, y=confidences, marker_color=colors_bar,
    text=[f"{c:.0%}<br>{s}" for c, s in zip(confidences, severities)],
    textposition='auto'
))
fig.update_layout(
    title='Few-Shot Risk Classification Results',
    xaxis_title='Risk Category', yaxis_title='Classification Confidence',
    yaxis=dict(range=[0, 1.1], tickformat='.0%'), width=750, height=450
)
fig.show()

print("\n💡 Few-Shot의 핵심 가치: 복잡한 지시 없이 예시만으로 분류 패턴을 전달할 수 있다.")
print("   조직 내 표준화된 리스크 분류 체계를 AI에게 빠르게 학습시킬 수 있다.")

In [ ]:
# === ReAct 패턴 시뮬레이션: 시장 분석 에이전트 ===

print("🤖 ReAct Agent: 전기차 충전 인프라 시장 분석")
print("="*70)

react_trace = [
    {"step": 1, "thought": "시장 규모와 성장률을 먼저 파악해야 한다.",
     "action": "search_market('전기차 충전 인프라')",
     "observation": "시장 2조원(2024), CAGR 25%, 2027년 5.2조원",
     "tool": "search_market", "time": 2.1},
    {"step": 2, "thought": "시장 규모를 확인했다. 경쟁 현황을 파악해야 한다.",
     "action": "query_competitors('EV charging')",
     "observation": "한국전력(35%), SK E&S(20%), GS칼텍스(15%), 기타(30%)",
     "tool": "query_competitors", "time": 1.8},
    {"step": 3, "thought": "상위 3사가 70% 점유. 규제 환경이 진입에 영향을 줄 수 있다.",
     "action": "get_regulations('EV infrastructure')",
     "observation": "2025년 충전 인프라 의무화 예정, 보조금 정책 강화",
     "tool": "get_regulations", "time": 1.5},
    {"step": 4, "thought": "규제가 시장 성장을 촉진할 것이다. 투자 수익성을 계산하자.",
     "action": "calculate_roi(investment=50, revenue_year3=15.6)",
     "observation": "ROI 15.3%, 회수기간 4.2년",
     "tool": "calculate_roi", "time": 0.8},
    {"step": 5, "thought": "충분한 정보를 수집했다. 종합 결론을 도출하자.",
     "action": "final_answer()",
     "observation": "시장 진입 유망. 틈새시장(급속충전 특화) 공략 권고.",
     "tool": "final_answer", "time": 1.2}
]

for step in react_trace:
    print(f"\n--- Step {step['step']} ({step['time']}s) ---")
    print(f"  💭 Thought:     {step['thought']}")
    print(f"  ⚡ Action:      {step['action']}")
    print(f"  👁 Observation: {step['observation']}")

total_time = sum(s['time'] for s in react_trace)
print(f"\n{'='*70}")
print(f"✅ ReAct 완료: 5단계, 총 {total_time:.1f}초")

# 타임라인 시각화
cumulative = [0]
for s in react_trace[:-1]:
    cumulative.append(cumulative[-1] + s['time'])

colors_map = {'search_market': '#636EFA', 'query_competitors': '#EF553B',
              'get_regulations': '#00CC96', 'calculate_roi': '#AB63FA', 'final_answer': '#FFA15A'}

fig = go.Figure()
for i, step in enumerate(react_trace):
    fig.add_trace(go.Bar(
        x=[step['time']], y=[f"Step {step['step']}: {step['tool']}"],
        orientation='h', base=cumulative[i],
        marker_color=colors_map[step['tool']],
        text=f"{step['time']}s", textposition='auto',
        name=step['tool'], showlegend=(i < 5)
    ))

fig.update_layout(title='ReAct Agent Execution Timeline',
    xaxis_title='Time (seconds)', width=850, height=400, barmode='stack', showlegend=True)
fig.show()

print("\n💡 ReAct 패턴의 핵심: 추론(Thought)과 행동(Action)을 교차하며")
print("   각 단계의 결과(Observation)를 관찰한 후 다음 행동을 결정한다.")
print("   → 추론 과정이 투명하여 감사(Audit)가 필요한 의사결정에 적합하다.")

In [ ]:
# === 에이전트 패턴 비교 ===

patterns = {
    'ReAct': {'Flexibility': 4.5, 'Cost Efficiency': 3.5, 'Quality': 4.0, 'Speed': 3.0, 'Transparency': 5.0},
    'Plan-Execute': {'Flexibility': 3.0, 'Cost Efficiency': 4.0, 'Quality': 4.2, 'Speed': 4.0, 'Transparency': 4.0},
    'Reflexion': {'Flexibility': 4.0, 'Cost Efficiency': 2.5, 'Quality': 4.8, 'Speed': 2.0, 'Transparency': 4.5},
    'Multi-Agent': {'Flexibility': 5.0, 'Cost Efficiency': 2.0, 'Quality': 4.5, 'Speed': 3.5, 'Transparency': 3.5},
}

dims = list(list(patterns.values())[0].keys())
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

fig = go.Figure()
for i, (pattern, scores) in enumerate(patterns.items()):
    values = [scores[d] for d in dims] + [scores[dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=dims + [dims[0]],
        fill='toself', name=pattern, line_color=colors[i]
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5.5])),
    title='Agent Pattern Comparison (5 Dimensions)',
    width=750, height=550
)
fig.show()

print("💡 에이전트 패턴 선택 가이드:")
print("="*55)
print("• ReAct:        투명성 최고 → 감사 필요 시, 탐색적 분석")
print("• Plan-Execute: 속도/비용 균형 → 구조화된 정기 분석")
print("• Reflexion:    품질 최고 → 중요 의사결정, 시간 여유 시")
print("• Multi-Agent:  유연성 최고 → 복잡한 다분야 프로젝트")
print("\n→ 기획 업무에서는 보고서 품질이 중요하므로")
print("  Reflexion 또는 Multi-Agent 패턴이 권장됩니다.")

### 📝 이론 과제 16-2 (15분)

#### 과제: 고급 프롬프팅과 AI 에이전트

**질문:**

1. **Chain-of-Thought(CoT)와 Few-Shot Learning의 차이**를 설명하세요. 각각 어떤 기획 업무 상황에 더 적합한지 예시와 함께 3-4문장으로 서술하세요.

2. **ReAct 패턴의 핵심 구조**(Thought-Action-Observation)를 설명하고, 기획 업무에서 이 패턴이 유용한 이유를 2-3문장으로 서술하세요.

3. **본인이 수행하는 기획 업무에 가장 적합한 에이전트 패턴**을 하나 선택하고, 그 이유를 구체적으로 설명하세요. (ReAct, Plan-Execute, Reflexion, Multi-Agent 중 선택)

**조사 키워드:**
- "Chain of Thought prompting examples"
- "ReAct agent pattern LLM"
- "multi-agent system planning"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 16-2 제출란

**학번:** ___________  
**이름:** ___________

#### 1. CoT와 Few-Shot의 차이 및 적합한 상황

(여기에 작성)

#### 2. ReAct 패턴의 핵심 구조 설명

(여기에 작성)

#### 3. 가장 적합한 에이전트 패턴과 이유

(여기에 작성)

---
## ☕ 휴식 (15분)
15분 휴식 후 **Part 3**에서 계속됩니다.

---

---
## Part 3: RAG와 인간-AI 협업

### 16.4 RAG(Retrieval-Augmented Generation) 아키텍처

RAG는 LLM의 3대 한계를 극복하는 핵심 기술이다:

| 한계 | 문제 | RAG 해결 방식 |
|------|------|---------------|
| **지식 단절** | 학습 시점 이후 정보 모름 | 최신 문서를 검색하여 제공 |
| **환각(Hallucination)** | 사실과 다른 정보 생성 | "문서에 근거하여 답변하라" |
| **출처 부재** | 근거 없는 주장 | 답변 + 참조 문서 ID 반환 |

### RAG 3단계 파이프라인

```
[1. Indexing]  문서 수집 → 청크 분할 → 임베딩 → 벡터 DB 저장
[2. Retrieval] 질문 임베딩 → 유사도 검색 → 상위 k개 문서 반환
[3. Generation] 검색 문서 + 질문 → LLM → 답변 + 출처
```

### 핵심 기술 요소

- **임베딩**: 텍스트를 고차원 벡터로 변환 (의미 유사 → 벡터 거리 근접)
- **벡터 DB**: FAISS(로컬), ChromaDB(오픈소스), Pinecone(클라우드)
- **청크 분할**: 500-1000 토큰, 50-100 토큰 오버랩
- **검색 전략**: Semantic Search, Hybrid Search(+BM25), Re-ranking

### 16.5 Human-in-the-Loop (HITL) 설계

### 표 16.8 체크포인트 설계

| 단계 | 체크포인트 | 검토자 | 검토 내용 |
|------|------------|--------|----------|
| 환경분석 | 분석 검토 | 전략팀장 | 시장 데이터 정확성, 경쟁 분석 타당성 |
| 시나리오 | 가정 검증 | 재무팀 | 확률 추정, 재무 모델 검증 |
| 리스크 | 리스크 보완 | 리스크팀 | 누락 리스크, 대응 전략 |
| 보고서 | 경영진 승인 | CEO/CSO | 전략 방향, 투자 결정 |

### 인간-AI 협업 모델 4유형

| 모델 | AI 역할 | 인간 역할 | 적합 상황 |
|------|---------|-----------|----------|
| **AI 주도형** | 분석/초안 작성 | 검토/승인 | 반복적 분석 |
| **인간 주도형** | 지시 수행 | 방향 설정 | 전략적 결정 |
| **협업형** | 옵션 제시 | 선택/피드백 | 복잡한 프로젝트 |
| **감독형** | 자율 실행 | 예외 처리 | 정형화된 업무 |

### 16.6 윤리적 고려사항

- **편향 문제**: 학습 데이터의 편향이 분석에 반영될 수 있음
- **투명성**: AI의 권고 근거가 설명 가능해야 함
- **데이터 보안**: 기획 정보의 민감성, 보안 규정 준수
- **의존성 리스크**: AI 보조 도구로 활용하되, 핵심 판단 역량 유지

> **기획자의 역할 변화**: "분석 수행자" → "분석 오케스트레이터"  
> 올바른 질문을 던지고, AI를 조율하며, 출력을 검증하고, 최종 판단을 내리는 역할

In [ ]:
# === RAG 파이프라인 시뮬레이션 ===

print("🔍 RAG Pipeline Simulation: 전기차 충전 인프라 시장 분석")
print("="*65)

documents = {
    "DOC-001": {"title": "EV Charging Market Report 2024",
                "content": "전기차 충전 인프라 시장 규모는 2024년 2.1조원이며, 연평균 25% 성장하여 2027년 5.2조원에 달할 전망이다.",
                "source": "market_analysis_2024.md"},
    "DOC-002": {"title": "Competitor Analysis Q4 2024",
                "content": "한국전력(35%), SK E&S(20%), GS칼텍스(15%)가 상위 3사로 70%를 점유.",
                "source": "competitor_q4_2024.md"},
    "DOC-003": {"title": "EV Infrastructure Regulations",
                "content": "2025년부터 대규모 건물에 전기차 충전 설비 의무 설치. 정부 보조금 500억원 배정.",
                "source": "regulation_update.md"},
    "DOC-004": {"title": "Internal Strategy Memo",
                "content": "당사는 에너지 사업 경험을 활용하여 급속충전 특화 사업 모델을 검토 중.",
                "source": "internal_memo_2024.md"},
    "DOC-005": {"title": "Technology Trend Report",
                "content": "차세대 급속충전 기술(350kW)이 2026년 상용화 예정.",
                "source": "tech_trend_2024.md"},
}

print("\n[1단계] Indexing: 5개 문서 임베딩 완료")

sim_scores = {
    "DOC-001": 0.92, "DOC-002": 0.78, "DOC-003": 0.71,
    "DOC-005": 0.65, "DOC-004": 0.45,
}

query = "전기차 충전 시장의 규모와 성장 전망은?"
print(f"\n[2단계] Retrieval: 쿼리 = '{query}'")
print(f"\n{'순위':<4} {'문서ID':<10} {'유사도':<8} {'제목'}")
print("-"*65)
sorted_docs = sorted(sim_scores.items(), key=lambda x: x[1], reverse=True)
for rank, (doc_id, sim) in enumerate(sorted_docs, 1):
    indicator = "✅" if sim >= 0.7 else "⬜"
    print(f"{rank:<4} {doc_id:<10} {sim:.2f}    {indicator} {documents[doc_id]['title']}")

print(f"\n→ Top-3 문서를 LLM 컨텍스트로 전달")

print(f"\n[3단계] Generation:")
print(f"\n📝 생성된 답변:")
print(f"   전기차 충전 인프라 시장은 2024년 기준 2.1조원 규모이며,")
print(f"   연평균 25%의 높은 성장률을 보이고 있습니다. 2027년에는")
print(f"   5.2조원에 달할 전망입니다. 상위 3사(한국전력, SK E&S,")
print(f"   GS칼텍스)가 시장의 70%를 점유하고 있습니다.")
print(f"\n📎 출처: [DOC-001] market_analysis_2024.md (유사도 0.92)")
print(f"        [DOC-002] competitor_q4_2024.md (유사도 0.78)")
print(f"        [DOC-003] regulation_update.md (유사도 0.71)")

doc_ids = [d[0] for d in sorted_docs]
sims = [d[1] for d in sorted_docs]
titles = [documents[d[0]]['title'][:30] for d in sorted_docs]
bar_colors = ['#636EFA' if s >= 0.7 else '#D3D3D3' for s in sims]

fig = go.Figure(go.Bar(
    x=sims, y=[f"{did}: {t}" for did, t in zip(doc_ids, titles)],
    orientation='h', marker_color=bar_colors,
    text=[f"{s:.2f}" for s in sims], textposition='auto'
))
fig.add_vline(x=0.7, line_dash="dash", line_color="red", annotation_text="Threshold (0.70)")
fig.update_layout(title='RAG Retrieval: Document Similarity Scores',
    xaxis_title='Cosine Similarity', xaxis=dict(range=[0, 1.05]),
    width=800, height=400)
fig.show()

print("\n💡 RAG의 핵심 가치: 최신 정보 + 환각 방지 + 출처 추적으로 기획 보고서의 신뢰성을 확보한다.")

In [ ]:
# === 워크플로우 단계별 신뢰도 평가 ===

stages = {
    'Environment Analysis': {'confidence': 0.92, 'reviewer': '전략팀장', 'items': 12},
    'Scenario Planning': {'confidence': 0.88, 'reviewer': '재무팀', 'items': 8},
    'Risk Assessment': {'confidence': 0.90, 'reviewer': '리스크팀', 'items': 6},
    'Report Generation': {'confidence': 0.95, 'reviewer': 'CEO/CSO', 'items': 15},
}

avg_confidence = np.mean([s['confidence'] for s in stages.values()])

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=list(stages.keys())
)

positions = [(1,1), (1,2), (2,1), (2,2)]
for (stage_name, info), (r, c) in zip(stages.items(), positions):
    color = '#00CC96' if info['confidence'] >= 0.9 else '#FFA15A'
    fig.add_trace(go.Indicator(
        mode="gauge+number", value=info['confidence'] * 100,
        number={'suffix': '%'},
        gauge={'axis': {'range': [0, 100]}, 'bar': {'color': color},
               'steps': [{'range': [0, 70], 'color': '#FFE0E0'},
                         {'range': [70, 85], 'color': '#FFF3E0'},
                         {'range': [85, 100], 'color': '#E0FFE0'}],
               'threshold': {'line': {'color': 'red', 'width': 2}, 'thickness': 0.75, 'value': 80}}
    ), row=r, col=c)

fig.update_layout(title=f'Workflow Stage Confidence Scores (Average: {avg_confidence:.1%})',
    width=800, height=600)
fig.show()

print(f"📊 워크플로우 신뢰도 평가 결과")
print(f"{'='*55}")
print(f"{'단계':<22} {'신뢰도':<10} {'검토자':<12} {'분석 항목'}")
print(f"{'-'*55}")
for stage, info in stages.items():
    status = "✅" if info['confidence'] >= 0.9 else "⚠️"
    print(f"{stage:<22} {info['confidence']:.0%}       {info['reviewer']:<12} {info['items']}건 {status}")
print(f"{'-'*55}")
print(f"{'평균 신뢰도':<22} {avg_confidence:.1%}")
print(f"\n💡 모든 단계의 신뢰도가 80% 임계값을 초과하여 워크플로우가 정상 완료되었습니다.")
print(f"   단, 시나리오 기획(88%)은 상대적으로 낮아 재무팀의 면밀한 검토가 권장됩니다.")

In [ ]:
# === Human-in-the-Loop 워크플로우 시각화 (Sankey Diagram) ===

labels = [
    "Project Brief", "Env Analysis Agent", "Checkpoint 1", "Strategy Team Lead",
    "Scenario Agent", "Checkpoint 2", "Finance Team",
    "Risk Agent", "Checkpoint 3", "Risk Team",
    "Report Agent", "Checkpoint 4", "CEO/CSO",
    "Approved ✅", "Revision ↩️"
]

sources = [0, 1, 2, 2, 3, 4, 5, 5, 6, 7, 8, 8, 9, 10, 11, 11, 12, 14]
targets = [1, 2, 3, 14, 4, 5, 6, 14, 7, 8, 9, 14, 10, 11, 12, 14, 13, 1]
values =  [100, 100, 92, 8, 92, 92, 85, 7, 85, 85, 80, 5, 80, 80, 75, 5, 75, 25]

colors_link = [
    '#636EFA', '#636EFA', '#00CC96', '#EF553B',
    '#00CC96', '#636EFA', '#00CC96', '#EF553B',
    '#00CC96', '#636EFA', '#00CC96', '#EF553B',
    '#00CC96', '#636EFA', '#00CC96', '#EF553B',
    '#00CC96', '#FFA15A'
]

node_colors = [
    '#636EFA', '#AB63FA', '#FFA15A', '#00CC96',
    '#AB63FA', '#FFA15A', '#00CC96',
    '#AB63FA', '#FFA15A', '#00CC96',
    '#AB63FA', '#FFA15A', '#00CC96',
    '#19D3F3', '#EF553B'
]

fig = go.Figure(go.Sankey(
    node=dict(pad=15, thickness=20, label=labels, color=node_colors),
    link=dict(source=sources, target=targets, value=values, color=colors_link)
))
fig.update_layout(title='Human-in-the-Loop Planning Workflow (Sankey Diagram)',
    width=950, height=550)
fig.show()

print("💡 Human-in-the-Loop 워크플로우 해석:")
print("="*55)
print("• 각 AI 에이전트 실행 후 체크포인트에서 인간 검토자가 검증")
print("• 승인(✅) 시 다음 단계로 진행, 반려(↩️) 시 수정 후 재실행")
print("• 4개 체크포인트: 전략팀장 → 재무팀 → 리스크팀 → CEO/CSO")
print("\n→ AI가 분석을 수행하고, 인간이 검증·결정하는 '협업형' 모델입니다.")

In [ ]:
# === 인간-AI 협업 모델 비교 ===

models = {
    'AI-Led': {'AI Autonomy': 4.5, 'Human Involvement': 2.0, 'Quality': 3.5, 'Speed': 4.8, 'Cost Efficiency': 4.5},
    'Human-Led': {'AI Autonomy': 1.5, 'Human Involvement': 5.0, 'Quality': 4.0, 'Speed': 2.0, 'Cost Efficiency': 2.5},
    'Collaborative': {'AI Autonomy': 3.0, 'Human Involvement': 3.5, 'Quality': 4.8, 'Speed': 3.0, 'Cost Efficiency': 3.5},
    'Supervisory': {'AI Autonomy': 4.0, 'Human Involvement': 2.5, 'Quality': 4.2, 'Speed': 4.0, 'Cost Efficiency': 4.0},
}

dims = list(list(models.values())[0].keys())
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

fig = go.Figure()
for i, (model, scores) in enumerate(models.items()):
    values = [scores[d] for d in dims] + [scores[dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=dims + [dims[0]],
        fill='toself', name=model, line_color=colors[i]
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5.5])),
    title='Human-AI Collaboration Models Comparison',
    width=800, height=550
)
fig.show()

print("💡 협업 모델 선택 가이드:")
print("="*55)
print("• AI 주도형:   반복적 정기 보고서, 데이터 모니터링 → 속도/비용 우선")
print("• 인간 주도형: CEO 직접 지시 전략, 정치적 판단 필요 → 품질/통제 우선")
print("• 협업형:      복잡한 신규 프로젝트, 다관점 분석 → 품질 최고 ⭐")
print("• 감독형:      정형화된 월간 분석, 이상 감지 → 효율성/자율성 균형")
print("\n→ 기획 업무에서는 '협업형'이 가장 적합합니다.")
print("  AI가 옵션을 제시하고, 인간이 맥락을 고려하여 결정합니다.")

### 📝 이론 과제 16-3 (15분)

#### 과제: RAG와 인간-AI 협업

**질문:**

1. **RAG(Retrieval-Augmented Generation)가 필요한 이유** 3가지를 각각 2-3문장으로 설명하세요. (지식 단절, 환각 방지, 출처 추적 관점)

2. **Human-in-the-Loop 체크포인트 설계 원칙**을 설명하세요. "모든 단계에서 검토"가 아닌 "핵심 의사결정 지점에서 검토"가 중요한 이유를 2-3문장으로 서술하세요.

3. **AI 기획 시스템 도입 시 윤리적 고려사항**을 3가지 이상 나열하고, 각각에 대한 대응 방안을 제시하세요.

**조사 키워드:**
- "RAG retrieval augmented generation architecture"
- "human in the loop AI system design"
- "AI ethics enterprise planning"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 16-3 제출란

**학번:** ___________  
**이름:** ___________

#### 1. RAG가 필요한 이유 3가지

(여기에 작성)

#### 2. HITL 체크포인트 설계 원칙

(여기에 작성)

#### 3. 윤리적 고려사항과 대응 방안

(여기에 작성)

---
## Part 4: 실습 - 프롬프트 품질 평가와 CoT/Few-Shot 구현

이 파트에서는 앞서 배운 이론을 **직접 코드로 구현**합니다:
1. 다양한 프롬프트를 점수화하고 비교하는 완전한 평가 시스템
2. Chain-of-Thought를 활용한 다단계 재무 분석
3. Few-Shot 기반 리스크 분류 시스템

In [ ]:
# === 실습 4-1: 완전한 프롬프트 평가 시스템 ===

def evaluate_prompt_detailed(prompt_text):
    """프롬프트를 5개 차원으로 상세 평가"""
    scores = {}
    role_kw = ['역할', '당신은', 'You are', '전문가', '컨설턴트', '분석가', '경력']
    scores['Role'] = min(5, 1 + sum(1 for kw in role_kw if kw in prompt_text) * 1.5)
    context_kw = ['시장', '규모', '배경', '현재', '조원', 'CAGR', '기업', '매출', '산업']
    scores['Context'] = min(5, 1 + sum(1 for kw in context_kw if kw in prompt_text) * 0.8)
    inst_kw = ['분석하세요', '평가하세요', '도출하세요', '비교하세요', '관점에서', '단계로']
    scores['Instruction'] = min(5, 1 + sum(1 for kw in inst_kw if kw in prompt_text) * 1.5)
    const_kw = ['제약', '기반', '제한', '가정', '데이터', '객관적', '정량적', '이내']
    scores['Constraints'] = min(5, 1 + sum(1 for kw in const_kw if kw in prompt_text) * 1.2)
    format_kw = ['표', 'JSON', '형식', '목록', '구조', '항목', '정리', '요약']
    scores['Format'] = min(5, 1 + sum(1 for kw in format_kw if kw in prompt_text) * 1.2)
    return {k: round(v, 1) for k, v in scores.items()}

prompts = {
    'Basic': "시장 분석해줘",
    'Intermediate': "당신은 전략 컨설턴트입니다. 전기차 충전 시장을 분석하세요. 표 형식으로 정리해주세요.",
    'Advanced': """[역할] 당신은 10년 경력의 전략 컨설턴트입니다.
[맥락] 국내 전기차 충전 인프라 시장 규모 2조원, CAGR 25%.
       분석 주체: 중견 에너지 기업 (매출 5천억, 충전 사업 미경험).
[지시사항] 다음 3가지 관점에서 분석하세요: 1) 성장 동인 2) 경쟁 구도 3) 기회 영역
[제약조건] 데이터 기반 정량적 분석 우선. 불확실한 가정은 명시.
[출력형식] 각 관점별 핵심 발견을 표 형식으로 정리하고, 최종 권고를 3가지 항목으로 요약"""
}

results = {}
for level, prompt in prompts.items():
    results[level] = evaluate_prompt_detailed(prompt)

dims = ['Role', 'Context', 'Instruction', 'Constraints', 'Format']
colors_r = ['#EF553B', '#FFA15A', '#636EFA']

fig = go.Figure()
for i, (level, scores) in enumerate(results.items()):
    vals = [scores[d] for d in dims] + [scores[dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=dims + [dims[0]],
        fill='toself', name=level, line_color=colors_r[i]
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 6])),
    title='Prompt Quality: Basic vs Intermediate vs Advanced',
    width=750, height=550
)
fig.show()

print("📊 프롬프트 품질 평가 비교")
print(f"{'='*60}")
print(f"{'차원':<15} {'Basic':<10} {'Intermediate':<15} {'Advanced':<10}")
print(f"{'-'*60}")
for dim in dims:
    print(f"{dim:<15} {results['Basic'][dim]:<10} {results['Intermediate'][dim]:<15} {results['Advanced'][dim]:<10}")
totals = {level: sum(scores.values()) for level, scores in results.items()}
print(f"{'-'*60}")
print(f"{'총점':<15} {totals['Basic']:<10.1f} {totals['Intermediate']:<15.1f} {totals['Advanced']:<10.1f}")
imp = (totals['Advanced'] - totals['Basic']) / totals['Basic'] * 100
print(f"\n💡 Advanced 프롬프트는 Basic 대비 {imp:.0f}% 높은 품질 점수를 기록했습니다.")

In [ ]:
# === 실습 4-2: CoT 기반 다단계 재무 분석 ===

print("🔗 Chain-of-Thought: 신규 사업 투자 타당성 분석")
print("="*65)

project = {
    'name': '전기차 충전 인프라 사업', 'investment': 50,
    'revenue_year1': 8, 'revenue_year2': 18, 'revenue_year3': 35,
    'cost_year1': 15, 'cost_year2': 20, 'cost_year3': 25,
    'discount_rate': 0.10
}

print("\n--- Step 1: 연도별 영업이익 계산 ---")
profits = []
for y in range(1, 4):
    rev = project[f'revenue_year{y}']
    cost = project[f'cost_year{y}']
    profit = rev - cost
    margin = profit / rev * 100 if rev > 0 else 0
    profits.append(profit)
    print(f"  Year {y}: 매출 {rev}억 - 비용 {cost}억 = 영업이익 {profit}억원 (마진 {margin:.1f}%)")

print(f"\n--- Step 2: NPV (순현재가치) 계산 ---")
r = project['discount_rate']
npv = -project['investment']
pv_list = []
for y, profit in enumerate(profits, 1):
    pv = profit / (1 + r) ** y
    pv_list.append(pv)
    npv += pv
    print(f"  Year {y}: PV = {profit} / (1+{r})^{y} = {pv:.2f}억원")
print(f"  NPV = -{project['investment']} + {sum(pv_list):.2f} = {npv:.2f}억원")

print(f"\n--- Step 3: 회수기간 (Payback Period) 계산 ---")
cumulative = 0
payback = None
for y, profit in enumerate(profits, 1):
    cumulative += profit
    print(f"  Year {y}: 누적 이익 = {cumulative}억원 (투자 {project['investment']}억원)")
    if cumulative >= project['investment'] and payback is None:
        prev_cum = cumulative - profit
        remaining = project['investment'] - prev_cum
        fraction = remaining / profit
        payback = y - 1 + fraction
        print(f"  → 회수 시점: Year {payback:.1f}")
if payback is None:
    payback = 3 + (project['investment'] - cumulative) / profits[-1]
    print(f"  → 3년 내 미회수, 예상 회수: Year {payback:.1f}")

print(f"\n--- Step 4: ROI 계산 ---")
total_profit = sum(profits)
roi = (total_profit - project['investment']) / project['investment'] * 100
print(f"  총 이익 = {total_profit}억원")
print(f"  ROI = ({total_profit} - {project['investment']}) / {project['investment']} = {roi:.1f}%")

print(f"\n--- Step 5: 종합 판단 ---")
decision = "투자 권고 ✅" if npv > 0 and roi > 0 else "투자 재검토 ⚠️"
print(f"  NPV: {npv:.2f}억원 ({'양수 → 가치 창출' if npv > 0 else '음수 → 가치 파괴'})")
print(f"  ROI: {roi:.1f}%")
print(f"  회수기간: {payback:.1f}년")
print(f"  → 결론: {decision}")

fig = make_subplots(rows=1, cols=2, subplot_titles=('Annual Revenue, Cost & Profit', 'Cumulative Cash Flow'))
years = ['Year 1', 'Year 2', 'Year 3']
revs = [project[f'revenue_year{y}'] for y in range(1, 4)]
costs = [project[f'cost_year{y}'] for y in range(1, 4)]

fig.add_trace(go.Bar(name='Revenue', x=years, y=revs, marker_color='#636EFA'), row=1, col=1)
fig.add_trace(go.Bar(name='Cost', x=years, y=costs, marker_color='#EF553B'), row=1, col=1)
fig.add_trace(go.Bar(name='Profit', x=years, y=profits, marker_color='#00CC96'), row=1, col=1)

cum_cf = [-project['investment']]
for p in profits:
    cum_cf.append(cum_cf[-1] + p)
x_cf = ['Year 0', 'Year 1', 'Year 2', 'Year 3']
cf_colors = ['#EF553B' if v < 0 else '#00CC96' for v in cum_cf]
fig.add_trace(go.Bar(name='Cumulative CF', x=x_cf, y=cum_cf, marker_color=cf_colors, showlegend=False), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_layout(title='CoT Financial Analysis: Investment Viability', width=900, height=450, barmode='group')
fig.update_yaxes(title_text='Amount (Billion KRW)', row=1, col=1)
fig.update_yaxes(title_text='Cumulative (Billion KRW)', row=1, col=2)
fig.show()

In [ ]:
# === 실습 4-3: Few-Shot 리스크 분류 시스템 ===

few_shot_examples = [
    {"text": "핵심 개발자 3명이 경쟁사로 이직했습니다.", "category": "HR", "severity": 4},
    {"text": "주요 원자재 가격이 30% 상승했습니다.", "category": "Financial", "severity": 3},
    {"text": "신규 개인정보보호법이 다음 분기 시행됩니다.", "category": "Regulatory", "severity": 3},
    {"text": "주요 협력사가 파산 신청을 했습니다.", "category": "Supply Chain", "severity": 5},
    {"text": "경쟁사가 유사 제품을 50% 저렴하게 출시했습니다.", "category": "Market", "severity": 4},
]

test_cases = [
    {"text": "CTO가 갑작스럽게 사직서를 제출했습니다.", "expected": "HR"},
    {"text": "환율이 15% 급등하여 수입 비용이 증가했습니다.", "expected": "Financial"},
    {"text": "EU AI 규제법이 우리 제품에 적용됩니다.", "expected": "Regulatory"},
    {"text": "반도체 공급 부족으로 납기가 3개월 지연됩니다.", "expected": "Supply Chain"},
    {"text": "시장 점유율 1위 기업이 우리 고객사를 공략하고 있습니다.", "expected": "Market"},
]

def classify_risk_fewshot(text, examples):
    """Few-Shot 패턴 매칭 시뮬레이션"""
    keyword_map = {
        'HR': ['사직', '이직', '인재', '퇴사', 'CTO', '개발자'],
        'Financial': ['환율', '가격', '비용', '매출', '재무', '손실'],
        'Regulatory': ['규제', '법', '컴플라이언스', 'EU', '개인정보'],
        'Supply Chain': ['공급', '납기', '협력사', '파산', '반도체'],
        'Market': ['경쟁사', '시장', '점유율', '고객', '가격 경쟁'],
    }
    best_cat, best_score = 'Unknown', 0
    for cat, keywords in keyword_map.items():
        score = sum(1 for kw in keywords if kw in text)
        if score > best_score:
            best_score = score
            best_cat = cat
    confidence = min(0.95, 0.7 + best_score * 0.1)
    severity = np.random.choice([3, 4, 5], p=[0.3, 0.5, 0.2])
    return best_cat, confidence, severity

print("📋 Few-Shot Risk Classification System")
print("="*65)
print(f"학습 예시: {len(few_shot_examples)}건 | 테스트: {len(test_cases)}건")

results_fs = []
correct = 0
for i, test in enumerate(test_cases, 1):
    pred_cat, conf, sev = classify_risk_fewshot(test['text'], few_shot_examples)
    match = pred_cat == test['expected']
    correct += int(match)
    results_fs.append({'id': i, 'text': test['text'][:35], 'expected': test['expected'],
                       'predicted': pred_cat, 'confidence': conf, 'match': match})
    status = "✅" if match else "❌"
    print(f"\n  Test {i}: {test['text'][:50]}")
    print(f"    예상: {test['expected']} | 예측: {pred_cat} (신뢰도: {conf:.0%}) {status}")

accuracy = correct / len(test_cases)
print(f"\n{'='*65}")
print(f"📊 분류 정확도: {accuracy:.0%} ({correct}/{len(test_cases)})")

cat_colors = {'HR': '#636EFA', 'Financial': '#EF553B', 'Regulatory': '#00CC96',
              'Supply Chain': '#AB63FA', 'Market': '#FFA15A'}

fig = go.Figure()
for r in results_fs:
    fig.add_trace(go.Bar(
        x=[r['predicted']], y=[r['confidence']],
        name=f"Test {r['id']}: {r['text'][:20]}...",
        marker_color=cat_colors.get(r['predicted'], '#999'),
        text=f"Test {r['id']}<br>{r['confidence']:.0%}", textposition='auto'
    ))

fig.update_layout(
    title=f'Few-Shot Risk Classification Results (Accuracy: {accuracy:.0%})',
    xaxis_title='Risk Category', yaxis_title='Confidence',
    yaxis=dict(range=[0, 1.1], tickformat='.0%'),
    width=850, height=450, showlegend=False
)
fig.show()

print(f"\n💡 Few-Shot은 5개 예시만으로 {accuracy:.0%} 정확도를 달성했습니다.")
print("   조직의 리스크 분류 체계를 AI에게 빠르게 전달할 수 있는 효율적 기법입니다.")

### 📝 실습 과제 16-4 (25분)

#### 과제: 프롬프트 평가 시스템 활용

**목표:** 본인의 업무/관심 분야에 대한 프롬프트 3개(기본/중급/고급)를 작성하고, `evaluate_prompt_detailed()` 함수로 평가한 후 레이더 차트로 비교하세요.

**요구사항:**
1. **기본 프롬프트**: 1문장 이내, 요소 미포함
2. **중급 프롬프트**: 2-3개 요소 포함
3. **고급 프롬프트**: 5개 요소 모두 포함 (역할/맥락/지시/제약/형식)
4. `evaluate_prompt_detailed()` 함수로 각각 평가
5. Plotly 레이더 차트로 3개 프롬프트 비교 시각화
6. 결과 해석 (print문으로 2-3줄)

**제출:** 아래 코드 셀에 작성 후, 실행 결과와 함께 제출

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO: 1. 기획 업무용 프롬프트 3개 작성 (기본/중급/고급)
my_prompts = {
    'basic': "",        # 여기에 기본 프롬프트 작성 (예: "매출 분석해줘")
    'intermediate': "", # 여기에 중급 프롬프트 작성 (2-3개 요소 포함)
    'advanced': "",     # 여기에 고급 프롬프트 작성 (5개 요소 모두 포함)
}

# TODO: 2. evaluate_prompt_detailed 함수로 각각 평가
# my_results = {}
# for level, prompt in my_prompts.items():
#     my_results[level] = evaluate_prompt_detailed(prompt)
#     print(f"{level}: {my_results[level]}")

# TODO: 3. Plotly 레이더 차트로 비교 시각화
# dims = ['Role', 'Context', 'Instruction', 'Constraints', 'Format']
# fig = go.Figure()
# ... (레이더 차트 코드 작성)
# fig.show()

# TODO: 4. 결과 해석 (2-3줄 print문)

# ========== 여기까지 ==========

### ✅ 실습 과제 16-4 제출란

**학번:** ___________  
**이름:** ___________

#### 작성한 프롬프트 3개

**기본:**

(여기에 작성)

**중급:**

(여기에 작성)

**고급:**

(여기에 작성)

#### 결과 해석

(여기에 작성)

---
## Part 5: 실습 - 기획 워크플로우 시뮬레이션

이 파트에서는 **4개 AI 에이전트가 협업하는 기획 워크플로우**를 시뮬레이션합니다:
1. 환경분석 에이전트 → 시장 데이터 수집
2. 시나리오 에이전트 → 3가지 시나리오 + 기대값 계산
3. 리스크 에이전트 → 리스크 레지스터 작성
4. 보고서 에이전트 → 종합 전략 보고서

각 단계에서 신뢰도 점수와 처리 시간을 추적하고, 최종 대시보드를 생성합니다.

In [ ]:
# === 실습 5-1: 4-에이전트 기획 워크플로우 시뮬레이션 ===

print("🏭 Multi-Agent Planning Workflow Simulation")
print("   프로젝트: 전기차 충전 인프라 시장 진입 전략")
print("="*65)

project_brief = {
    'name': '전기차 충전 인프라 시장 진입', 'industry': '전기차 충전',
    'budget': 50, 'timeline': '3년', 'objective': '신규 사업 진출 타당성 분석'
}

# Agent 1: 환경분석
print("\n🔍 [Agent 1] 환경분석 에이전트 실행 중...")
env_result = {
    'market_size': 21000, 'cagr': 0.25, 'market_2027': 52000,
    'competitors': {'한국전력': 0.35, 'SK E&S': 0.20, 'GS칼텍스': 0.15, '기타': 0.30},
    'key_insight': '규제가 시장 성장을 촉진, 급속충전 틈새시장에 기회 존재',
    'confidence': 0.92, 'processing_time': 3.2
}
print(f"   ✅ 완료 (신뢰도: {env_result['confidence']:.0%}, {env_result['processing_time']}s)")
print(f"   시장 규모: {env_result['market_size']:,}억원 → {env_result['market_2027']:,}억원 (2027)")
print(f"   핵심 인사이트: {env_result['key_insight']}")

# Agent 2: 시나리오
print("\n📊 [Agent 2] 시나리오 에이전트 실행 중...")
scenarios = {
    'Base (50%)': {'probability': 0.50, 'revenue': 156, 'description': '현 트렌드 지속'},
    'Optimistic (25%)': {'probability': 0.25, 'revenue': 350, 'description': '규제 강화 + 보조금 확대'},
    'Pessimistic (25%)': {'probability': 0.25, 'revenue': 70, 'description': '경쟁 심화 + 경기 둔화'},
}
expected_value = sum(s['probability'] * s['revenue'] for s in scenarios.values())
scenario_conf = 0.88
print(f"   ✅ 완료 (신뢰도: {scenario_conf:.0%}, 2.8s)")
for name, s in scenarios.items():
    print(f"   {name}: 매출 {s['revenue']}억원 ({s['description']})")
print(f"   기대값: {expected_value:.0f}억원")

# Agent 3: 리스크
print("\n⚠️ [Agent 3] 리스크 에이전트 실행 중...")
risks = pd.DataFrame({
    'Risk ID': ['R1', 'R2', 'R3', 'R4', 'R5'],
    'Description': ['대기업 가격 경쟁', '기술 표준 변경', '인허가 지연', '인력 확보 실패', '초기 손실 장기화'],
    'Probability': [0.7, 0.4, 0.3, 0.5, 0.4],
    'Impact': [4, 3, 3, 3, 4],
    'Risk Score': [2.8, 1.2, 0.9, 1.5, 1.6],
    'Mitigation': ['차별화 전략', '유연한 설계', '사전 협의', '인센티브 패키지', '단계적 투자']
})
risk_conf = 0.90
print(f"   ✅ 완료 (신뢰도: {risk_conf:.0%}, 2.1s)")
print(f"   식별된 리스크: {len(risks)}건 | 최고위험: R1 대기업 가격 경쟁 (점수 2.8)")

# Agent 4: 보고서
print("\n📝 [Agent 4] 보고서 에이전트 실행 중...")
report_phases = [
    '1단계 (0-12개월): 제휴 기반 파일럿 - 초기 투자 10억원',
    '2단계 (12-24개월): 틈새시장 확대 - 추가 투자 20억원',
    '3단계 (24-36개월): 본격 스케일업 - 추가 투자 20억원'
]
report_conf = 0.95
print(f"   ✅ 완료 (신뢰도: {report_conf:.0%}, 1.9s)")
print(f"   권고: 3단계 진입 전략")
for phase in report_phases:
    print(f"     • {phase}")

total_time = 3.2 + 2.8 + 2.1 + 1.9
avg_conf = np.mean([0.92, 0.88, 0.90, 0.95])
print(f"\n{'='*65}")
print(f"🏁 워크플로우 완료!")
print(f"   총 소요시간: {total_time:.1f}초 | 평균 신뢰도: {avg_conf:.1%}")
print(f"   검토 대기: 4건 (각 체크포인트에서 인간 검토 필요)")

In [ ]:
# === 실습 5-2: 워크플로우 결과 종합 대시보드 ===

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Pipeline Status & Confidence', 'Scenario Comparison (Revenue)',
                    'Risk Heatmap (Probability x Impact)', 'Phased Investment Plan'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "bar"}]]
)

# 1. Pipeline Status
agents = ['Env Analysis', 'Scenario', 'Risk', 'Report']
confidences_p = [0.92, 0.88, 0.90, 0.95]
proc_times = [3.2, 2.8, 2.1, 1.9]
conf_colors = ['#00CC96' if c >= 0.9 else '#FFA15A' for c in confidences_p]
fig.add_trace(go.Bar(
    x=agents, y=confidences_p, marker_color=conf_colors,
    text=[f"{c:.0%}<br>{t}s" for c, t in zip(confidences_p, proc_times)],
    textposition='auto', name='Confidence', showlegend=False
), row=1, col=1)
fig.add_hline(y=0.8, line_dash="dash", line_color="red", row=1, col=1)

# 2. Scenario Comparison
scen_names = ['Base (50%)', 'Optimistic (25%)', 'Pessimistic (25%)', 'Expected Value']
scen_values = [156, 350, 70, 183]
scen_colors = ['#636EFA', '#00CC96', '#EF553B', '#AB63FA']
fig.add_trace(go.Bar(
    x=scen_names, y=scen_values, marker_color=scen_colors,
    text=[f"{v}B" for v in scen_values], textposition='auto',
    name='Revenue', showlegend=False
), row=1, col=2)

# 3. Risk Scatter
risk_ids = ['R1', 'R2', 'R3', 'R4', 'R5']
risk_probs = [0.7, 0.4, 0.3, 0.5, 0.4]
risk_impacts = [4, 3, 3, 3, 4]
risk_scores = [p * i for p, i in zip(risk_probs, risk_impacts)]
risk_labels = ['Price War', 'Tech Change', 'Permit Delay', 'HR Failure', 'Loss Extension']

fig.add_trace(go.Scatter(
    x=risk_probs, y=risk_impacts, mode='markers+text',
    marker=dict(size=[s * 20 for s in risk_scores], color=risk_scores,
                colorscale='RdYlGn_r', showscale=False),
    text=risk_labels, textposition='top center',
    name='Risks', showlegend=False
), row=2, col=1)
fig.add_shape(type="rect", x0=0.5, y0=3.5, x1=1, y1=5,
              fillcolor="rgba(239,85,59,0.1)", line=dict(width=0), row=2, col=1)

# 4. Phased Investment
phases = ['Phase 1 (0-12M)', 'Phase 2 (12-24M)', 'Phase 3 (24-36M)']
investments = [10, 20, 20]
fig.add_trace(go.Bar(
    x=phases, y=investments, marker_color=['#636EFA', '#AB63FA', '#00CC96'],
    text=[f"{v}B KRW" for v in investments], textposition='auto',
    name='Investment', showlegend=False
), row=2, col=2)

fig.update_layout(
    title='Planning Workflow Dashboard: EV Charging Market Entry Strategy',
    width=1000, height=700, showlegend=False
)
fig.update_yaxes(title_text='Confidence', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Revenue (Billion KRW)', row=1, col=2)
fig.update_xaxes(title_text='Probability', row=2, col=1)
fig.update_yaxes(title_text='Impact (1-5)', row=2, col=1)
fig.update_yaxes(title_text='Investment (Billion KRW)', row=2, col=2)
fig.show()

print("💡 대시보드 해석:")
print("="*60)
print("1. 파이프라인: 모든 단계 신뢰도 80% 이상 → 워크플로우 정상 완료")
print("2. 시나리오: 기대값 183억원, 비관 시나리오에서도 70억원 → 하방 리스크 제한적")
print("3. 리스크: R1(대기업 가격경쟁)이 최고 위험 → 차별화 전략 필수")
print("4. 투자: 3단계 진입으로 리스크 분산 (10→20→20억원)")
print("\n→ 종합 판단: '3단계 점진적 진입' 전략을 권고합니다.")

### 📝 실습 과제 16-5 (25분)

#### 과제: 기획 워크플로우 설계

**목표:** 본인만의 **신규 프로젝트**에 대해 4-에이전트 기획 워크플로우를 설계하고 시뮬레이션하세요.

**요구사항:**
1. **프로젝트 브리프 정의**: 산업, 목표, 예산, 기간
2. **에이전트 역할 정의**: 최소 3개 에이전트 (환경분석, 시나리오, 리스크 등)
3. **각 에이전트의 시뮬레이션 출력**: 핵심 수치와 인사이트
4. **시나리오 3개 설계**: 기준/낙관/비관 + 기대값 계산
5. **Plotly 시각화**: 최소 2개 차트
6. **결과 해석**: print문으로 3-5줄

**프로젝트 예시:**
- 헬스케어 AI 솔루션 시장 진입
- 동남아 이커머스 진출
- 친환경 패키징 사업 전환
- 기업용 SaaS 플랫폼 개발

**제출:** 아래 코드 셀에 작성 후, 실행 결과와 함께 제출

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO: 1. 프로젝트 브리프 정의
my_project = {
    'name': '',        # 프로젝트명
    'industry': '',    # 산업
    'budget': 0,       # 예산 (억원)
    'timeline': '',    # 기간
    'objective': '',   # 목표
}

# TODO: 2. 에이전트 역할 정의 (최소 3개)
# agent_1_result = { 'name': '환경분석', 'confidence': 0.0, ... }
# agent_2_result = { 'name': '시나리오', 'confidence': 0.0, ... }
# agent_3_result = { 'name': '리스크', 'confidence': 0.0, ... }

# TODO: 3. 시나리오 3개 설계 + 기대값 계산
# my_scenarios = {
#     'Base (50%)': {'probability': 0.50, 'revenue': 0},
#     'Optimistic (25%)': {'probability': 0.25, 'revenue': 0},
#     'Pessimistic (25%)': {'probability': 0.25, 'revenue': 0},
# }
# expected_value = sum(s['probability'] * s['revenue'] for s in my_scenarios.values())

# TODO: 4. Plotly 시각화 (최소 2개 차트)
# fig = make_subplots(rows=1, cols=2, ...)
# fig.show()

# TODO: 5. 결과 해석 (3-5줄 print문)

# ========== 여기까지 ==========

### ✅ 실습 과제 16-5 제출란

**학번:** ___________  
**이름:** ___________

#### 프로젝트 개요

(여기에 작성)

#### 에이전트 역할 및 주요 결과

(여기에 작성)

#### 결과 해석 및 전략적 권고

(여기에 작성)

---
## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **구조화된 프롬프트**(역할/맥락/지시/형식)가 기본 대비 **272% 품질 향상** — 5가지 요소를 갖추면 AI 출력의 정확성, 구체성, 일관성이 크게 개선된다.
2. **CoT는 단계별 추론**, **Few-Shot은 예시 기반 학습**으로 분석 정확도 향상 — 재무 분석에는 CoT, 리스크 분류에는 Few-Shot이 효과적이다.
3. **AI 에이전트**는 도구 사용 + **ReAct 루프**(Thought-Action-Observation)로 복잡한 분석을 자율 수행 — 추론 과정이 투명하여 검증 가능하다.
4. **RAG**는 최신 정보 검색 + 환각 방지 + 출처 추적으로 기획 보고서의 **신뢰성을 확보**한다.
5. **Human-in-the-Loop**: AI가 분석하고 인간이 검증·결정하는 **협업 모델** — "무엇이 옳은가"의 판단은 여전히 인간의 몫이다.

### 핵심 메시지

> AI 에이전트는 기획의 전 주기를 지원하지만, **"무엇이 옳은가"**, **"무엇을 추구할 것인가"**의 판단은 여전히 인간의 몫이다. AI는 더 빠르고 정교한 분석을 가능하게 하지만, **최종 의사결정과 책임은 기획자에게** 있다.

### 📖 도서 전체 마무리

1-16장을 통해 기획의 전 과정을 다루었습니다:

| Part | 장 | 주제 |
|------|---|------|
| **Part 1-2** | 1-4장 | 기획 기초, 구조적 사고, 전략 프레임워크 |
| **Part 3** | 5-6장 | 인과추론, 시스템 다이내믹스 |
| **Part 4** | 7장 | AI 기반 환경 분석 (NLP, 감성분석) |
| **Part 5** | 8-10장 | 베이지안 사고, 시나리오 기획, 전략 시나리오 |
| **Part 6** | 11-13장 | 시뮬레이션, 다기준 의사결정, 실물옵션 |
| **Part 7** | 14-15장 | 실행 계획, 모니터링과 적응 |
| **Part 8** | 16장 | AI 에이전트와 기획 워크플로우 통합 |

> **AI와 인간이 협업하여 더 나은 의사결정을 내리는 것, 그것이 이 책이 지향하는 미래다.**

### 📚 추가 학습 자료

#### 참고문헌
- Wei, J., et al. (2022). Chain-of-Thought Prompting Elicits Reasoning in Large Language Models. *NeurIPS 2022*.
- Yao, S., et al. (2023). ReAct: Synergizing Reasoning and Acting in Language Models. *ICLR 2023*.
- Lewis, P., et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. *NeurIPS 2020*.
- Park, J. S., et al. (2023). Generative Agents: Interactive Simulacra of Human Behavior. *arXiv:2304.03442*.
- LangChain. (2024). *LangChain Documentation*. https://docs.langchain.com

#### Python 라이브러리
- `plotly`: 인터랙티브 시각화
- `numpy` / `pandas`: 데이터 처리 및 분석
- `langchain`: AI 에이전트 프레임워크 (참고)
- `openai` / `anthropic`: LLM API 클라이언트 (참고)
- `chromadb` / `faiss-cpu`: 벡터 데이터베이스 (참고)